# Lab 2.12 &mdash; Capstone: Visualizing How the Network Decides

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Introduction to Deep Learning**

### What you'll do
- Read a confusion matrix to see which digits get mixed up
- Compute per-class accuracy to find the model's weak spots
- Surface and view the digits the model gets wrong

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 2 slides &mdash; This connects to your labs](../../presentation/day1-module2-introduction-to-deep-learning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 2 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # quiet TensorFlow logs (used in the advanced labs)
WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-02-12")
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept &mdash; the module's payoff
A single accuracy number hides *how* a model decides. This capstone opens the box: a
**confusion matrix** (which true digit gets predicted as what), **per-class accuracy** (where it
struggles), and a look at the **misclassified** images themselves -- real error analysis.

> Needs `tensorflow`, `scikit-learn`, `matplotlib`. Target: MNIST; offline fallback to digits.

In [ ]:
# Data loader: real MNIST in the sandbox, sklearn digits as an offline fallback.
import numpy as np
def load_image_data(n_train=12000, n_test=2000):
    try:
        from tensorflow.keras.datasets import mnist
        (xtr, ytr), (xte, yte) = mnist.load_data()
        xtr = xtr.reshape(len(xtr), -1).astype("float32") / 255.0
        xte = xte.reshape(len(xte), -1).astype("float32") / 255.0
        name, side = "MNIST (28x28)", 28
    except Exception:
        from sklearn.datasets import load_digits
        d = load_digits(); X = (d.data / 16.0).astype("float32"); y = d.target
        xtr, ytr, xte, yte = X[:1400], y[:1400], X[1400:], y[1400:]
        name, side = "sklearn digits (8x8, offline fallback)", 8
    return (xtr[:n_train], ytr[:n_train]), (xte[:n_test], yte[:n_test]), name, side, xtr.shape[1]
from tensorflow import keras
from tensorflow.keras import layers

(X_tr, y_tr), (X_te, y_te), DATA_NAME, SIDE, NFEAT = load_image_data(n_train=10000, n_test=2000)
# train a quick classifier for us to inspect
model = keras.Sequential([layers.Input((NFEAT,)),
                          layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_tr, y_tr, epochs=3, batch_size=64, verbose=0)
import numpy as np
preds = model.predict(X_te, verbose=0).argmax(axis=1)
print("trained on", DATA_NAME, "| test samples:", len(y_te))

## Your Turn
Build the **confusion matrix**, compute **per-class accuracy**, and find the **misclassified** indices.

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

def conf_matrix():
    return ___   # TODO: confusion_matrix(y_te, preds)

def per_class_accuracy():
    cm = conf_matrix()
    return cm.diagonal() / ___   # TODO: divide by each row sum -> cm.sum(axis=1)

def misclassified_idx():
    return ___   # TODO: indices where preds != y_te  (np.where(...)[0])

try:
    pca = per_class_accuracy()
    print('confusion matrix shape:', conf_matrix().shape)
    print('weakest digit:', int(np.argmin(pca)), 'acc', round(float(pca.min()), 3))
    print('misclassified count:', len(misclassified_idx()))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

import numpy as np
expect_true("confusion matrix is 10x10 (ten digit classes)", lambda: conf_matrix().shape == (10, 10))
expect_true("per-class accuracy has 10 values in [0,1]", lambda: len(per_class_accuracy()) == 10 and all(0.0 <= a <= 1.0 for a in per_class_accuracy()))
expect_true("misclassified indices are consistent with accuracy", lambda: abs((1 - len(misclassified_idx())/len(y_te)) - (preds == y_te).mean()) < 1e-9)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Bonus: see the mistakes (not graded)
Visualise the confusion matrix and a few digits the model got wrong.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
    cm = conf_matrix(); wrong = misclassified_idx()
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    im = ax[0].imshow(cm, cmap="Blues"); ax[0].set_title("Confusion matrix")
    ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); fig.colorbar(im, ax=ax[0])
    if len(wrong):
        i = wrong[0]
        ax[1].imshow(X_te[i].reshape(SIDE, SIDE), cmap="gray_r")
        ax[1].set_title(f"pred {preds[i]} but true {y_te[i]}", color="red")
        ax[1].axis("off")
    plt.tight_layout(); plt.savefig(WORK + "/decisions.png", dpi=90); plt.show()
    print("saved:", WORK + "/decisions.png")
except Exception as e:
    print("Visualisation needs matplotlib + the blanks filled.", type(e).__name__)

---
### Key takeaway
You can now explain not just *how accurate* a model is but *how it decides* and *where it fails* -- the heart of responsible AI (Day 5). That completes Day 1.

[&#8592; All Module 2 labs](./index.html) &nbsp;&middot;&nbsp; [Module 2 slides](../../presentation/day1-module2-introduction-to-deep-learning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>